# Practical PyTorch · II · Part 1 — Companion Notebook

This goes with **"How a Model Learns: Being Wrong, On Purpose, A Million Times"**. Run the cells top to bottom (Shift+Enter).

No calculus here — just a tiny demo so you can *watch* the loss go down. We won't use a real neural network yet; we'll fit the simplest possible model so the moving parts (loss, gradient, optimizer, learning rate, epoch) are visible.

## 1. A toy problem the model has to learn

Pretend we don't know the rule behind some data. The true rule is `y = 2 * x` (the model doesn't get told this — it has to discover it). Our "model" is a single weight `w`, and its prediction is `w * x`. If training works, `w` should drift toward `2`.

In [ ]:
import torch

# Our data: inputs x, and the true answers y = 2 * x.
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y = torch.tensor([2.0, 4.0, 6.0, 8.0])

# Our entire "model": one weight, starting at a deliberately wrong value.
# requires_grad=True tells autograd to track this number so it can compute its gradient.
w = torch.tensor(0.5, requires_grad=True)

print("starting weight w:", w.item())

## 2. One single step, broken apart

Let's do exactly one nudge by hand so you can see each piece from the post:

1. **Predict** with the current weight.
2. Compute the **loss** — one number for how wrong we were.
3. Ask autograd for the **gradient** — which way to nudge `w`.
4. Take a step sized by the **learning rate**.

In [ ]:
learning_rate = 0.01

# 1. Predict.
prediction = w * x

# 2. Loss: average of (prediction - truth) squared. One number, lower = better.
loss = ((prediction - y) ** 2).mean()
print("loss before the step:", loss.item())

# 3. Autograd computes the gradient for w (the downhill direction).
loss.backward()
print("gradient on w:", w.grad.item())

# 4. Nudge w downhill by a small step. (no_grad: this bookkeeping isn't part of the model.)
with torch.no_grad():
    w -= learning_rate * w.grad
    w.grad.zero_()  # clear the gradient before the next step

print("weight after one step:", w.item())

`loss.backward()` is autograd doing the heavy lifting — computing the nudge direction for you. One step barely moved `w`. The whole trick of training is doing this many times.

## 3. Now loop it — and watch the loss fall

Same four steps, repeated. Each full pass over the data is an **epoch**. Watch the loss shrink and `w` climb toward the true value of `2`.

In [ ]:
# Start fresh.
w = torch.tensor(0.5, requires_grad=True)
learning_rate = 0.01

for epoch in range(30):
    prediction = w * x
    loss = ((prediction - y) ** 2).mean()

    loss.backward()                       # autograd: find the downhill direction
    with torch.no_grad():
        w -= learning_rate * w.grad       # optimizer step (done by hand here)
        w.grad.zero_()

    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss {loss.item():7.4f}  w {w.item():.4f}")

print("\nfinal w (true answer is 2.0):", round(w.item(), 4))

That's the whole post in one loop: predict, measure the loss, follow the gradient downhill, take a step, repeat. No one ever told the model the rule was `y = 2 * x` — it felt its way there in the fog.

## 4. Play with the learning rate

The single most important dial. Try the cell below with different values and see what happens:

- `0.001` — too small: it crawls and never quite arrives in 30 epochs.
- `0.01` — a sensible step.
- `0.3` — too big: watch the loss overshoot and blow up to `inf` / `nan`.

In [ ]:
def train(learning_rate, epochs=30):
    w = torch.tensor(0.5, requires_grad=True)
    for _ in range(epochs):
        loss = ((w * x - y) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= learning_rate * w.grad
            w.grad.zero_()
    return loss.item(), w.item()

for lr in [0.001, 0.01, 0.3]:
    final_loss, final_w = train(lr)
    print(f"lr {lr:<6} ->  final loss {final_loss:12.4f}   final w {final_w:.4f}")

Too small crawls, too big explodes, and there's a comfortable band in between. Finding that band is most of what "tuning" feels like in practice.

Next up (Part 2): the same loop, but written the real PyTorch way — with an actual optimizer like SGD or Adam doing the stepping instead of us doing it by hand.